In [0]:
%pip install pymongo

In [0]:
from pathlib import Path
from src.modules.data import data_factory
from src.modules.config import config_loader

environment: str = "dv"
layers_config: dict = config_loader(environment, Path("src/configs/layer_config.yaml"))
rule_exporter_config = config_loader(environment, Path("src/pipelines/rule_exporter/configuration/config.yaml"))

print("layers_config:", layers_config)
print("rule_exporter_config:", rule_exporter_config)

In [0]:
source_data_repository = data_factory(
    rule_exporter_config["source"],
    environment,
    layers_config,
    spark_session = spark
)

print(dir(source_data_repository))

In [0]:
from src.modules.metadata import extract_rule_metadata
import src.rules as r
import inspect

# Extract all rules metadata
rules_metadata = []
functions_in_r = [(name, obj) for name, obj in inspect.getmembers(r, inspect.isfunction)]

for name, obj in functions_in_r:
    metadata = extract_rule_metadata(obj)
    if metadata:
        rules_metadata.append(metadata)

print(f"📋 Found {len(rules_metadata)} rules to upsert")

In [0]:
source_data_repository.get_rules()

In [0]:
from src.modules.data import CatalogRepository

# Upsert rules to the target table
if rules_metadata:
    source_data_repository.upsert_rules(rules_metadata)
else:
    print("⚠️ No rules found to upsert")

In [0]:
source_data_repository.get_rules()